# Setup

## Import Necessary Dependancies

In [ ]:
# --- Data Deps --- #
import pandas as pd
import numpy as np

# --- Viz Deps --- #

# -- ML Deps --- #
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchmetrics import Accuracy, Precision, Recall, F1Score, MetricCollection

# --- Utils Deps --- #
from typing import Any, Optional
import os
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split

# --- MLFlow Deps --- #
import mlflow
from mlflow.models.signature import infer_signature

# --- TQDM Deps --- #
from tqdm.auto import tqdm

# --- Set Seed --- #
from ml_project_template.core.utils.utils import set_seed

## Load Environment Variables

In [ ]:
load_dotenv()

## Set Seed

In [ ]:
set_seed(42)

## Device Setup

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Setup MLFlow And Check Connection

In [ ]:
PROJECT_ROOT = os.path.abspath(
    os.path.join(os.getcwd(), "../../..")
)  # adjust if needed
MLFLOW_DB = os.path.join(PROJECT_ROOT, "mlflow.db")
ARTIFACT_ROOT = os.path.join(PROJECT_ROOT, "mlruns/")
experiment_name = "experiment-name"

mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")

existing_exp = mlflow.get_experiment_by_name(experiment_name)
if existing_exp is None:
    mlflow.create_experiment(
        name=experiment_name,
        artifact_location=f"file://{ARTIFACT_ROOT}",  # MUST be absolute path
    )

mlflow.pytorch.autolog()
assert mlflow.get_tracking_uri() == f"sqlite:///{MLFLOW_DB}", (
    "MLflow tracking URI is not set correctly!"
)
assert mlflow.get_experiment_by_name(experiment_name) is not None, (
    "MLflow experiment is not set correctly!"
)

## Read Dataset

In [ ]:
dataset_df = pd.read_csv("../../../data/raw/multiclass_rawdata.csv")
dataset_df.head()

# Data Preprocessing

## Train/Test/Validation Split

In [ ]:
X, y = dataset_df.drop("label", axis=1), dataset_df["label"]
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42
)

X_train.shape, X_val.shape, X_test.shape

## Preprocessing

Here, implement all data specific preprocessing.

# PyTorch DataLoader and Dataset

## Dataset

In [ ]:
class TestDataset(Dataset):  # type: ignore
    def __init__(self, features: np.ndarray, labels: np.ndarray):
        self.features = features
        self.labels = labels

    def __len__(self) -> int:
        return len(self.features)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return torch.tensor(self.features[idx], dtype=torch.float32), torch.tensor(
            self.labels[idx], dtype=torch.long
        )

## Prepare dataloader

In [ ]:
def data_loader(
    features: np.ndarray,
    labels: np.ndarray,
    batch_size: int = 64,
    shuffle: bool = False,
) -> DataLoader[Any]:
    dataset = TestDataset(features=features, labels=labels)
    pin_memory = True
    return DataLoader(
        dataset=dataset, batch_size=batch_size, shuffle=shuffle, pin_memory=pin_memory
    )  # type: ignore

In [ ]:
batch_size = 64
train_dl = data_loader(
    X_train.values, y_train.values, batch_size=batch_size, shuffle=True
)
val_dl = data_loader(X_val.values, y_val.values, batch_size=batch_size)
test_dl = data_loader(X_test.values, y_test.values, batch_size=batch_size)

## Get an sample from the data loaders

In [ ]:
X_train_example, y_train_example = next(iter(train_dl))
X_test_example, y_test_example = next(iter(test_dl))
X_val_example, y_val_example = next(iter(val_dl))

print(
    f"X_train_example.shape: {X_train_example.shape} | y_train_example.shape: {y_train_example.shape}"
)
print(
    f"X_test_example.shape: {X_test_example.shape}  | y_train_example.shape: {y_test_example.shape}"
)
print(
    f"X_val_example.shape: {X_val_example.shape}   | y_train_example.shape: {y_val_example.shape}"
)

# Model Setup

## Model Architecture

In [ ]:
class SimpleModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = torch.nn.Linear(2, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

## A test run to log the parameters of the model

In [ ]:
model = SimpleModel()
layer_summaries: list[dict[str, Any]] = []
total_params = 0


def count_model_parameters(model: nn.Module) -> int:
    def log_input_shape(
        module: torch.nn.Module, input: torch.Tensor, output: torch.Tensor
    ) -> None:
        name = module.__class__.__name__
        num_params = sum(p.numel() for p in module.parameters())
        layer_info: dict[str, Any] = {
            "layer": name,
            "input_shape": list(input[0].shape),
            "output_shape": list(output[0].shape),
            "num_params": num_params,
        }
        layer_summaries.append(layer_info)

    # Register hook on each layer
    for name, module in model.named_modules():  # type: ignore
        module.register_forward_hook(log_input_shape)  # type: ignore

    with torch.inference_mode():
        print(f"------- Input Shape -------\n{X_train_example.shape}")
        model(
            X_train_example
        )  # `img` is of shape [batch_size, num_channels, height, width]

    df = pd.DataFrame(layer_summaries)
    df.to_json("model_summary.json", orient="records", indent=4)
    return df


model_description_df = count_model_parameters(model=model)
model_description_df

## Early Stopping

In [ ]:
class EarlyStopping:
    """
    Early Stopping class

    Args:
        patience: int=5:
            The number of epochs to wait before deciding that the model is not learning anymore

        delta: float=0.0:
            Minimum change in monitored metric (loss in this implementation) that is considered improvement
    """

    def __init__(self, patience: int = 5, delta: float = 0.0):
        self.best_model_state = None
        self.patience = patience
        self.best_score = None
        self.early_stop = False
        self.delta = delta
        self.counter = 0

    def __call__(self, loss: float, model: torch.nn.Module) -> None:
        score = -loss

        if self.best_score is None:
            self.best_score = score
            self.best_model_state = {
                k: v.detach().clone() for k, v in model.state_dict().items()
            }

        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

    def load_best_model(self, model: torch.nn.Module) -> None:
        if self.best_model_state is None:
            raise ValueError(
                "Calling this method before running early stopping is not permitted"
            )
        model.load_state_dict(self.best_model_state)

    def reset(self) -> None:
        """Resets the early stopping state for a new training phase."""
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None

## Train-Test Loop

In [ ]:
def step(
    X: torch.Tensor,
    y: torch.Tensor,
    train: bool,
    loss_fn: nn.Module,
    model: nn.Module,
    optimizer: Optional[optim.Optimizer] = None,
    binary_decision_threshold: float = 0.5,
) -> tuple[float, torch.Tensor]:
    """Training or Training step for a binary classifier, returns (loss, processed_predictions)"""

    context = torch.enable_grad() if train else torch.inference_mode()
    with context:
        logits = model(X)
        loss = loss_fn(input=logits, target=y)

        if train:
            assert optimizer is not None, "Optimizer must be provided for training step"
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

    probs = torch.sigmoid(logits)
    preds = probs > binary_decision_threshold

    return loss.item(), preds


def train_loop(
    model: nn.Module,
    loss_fn: nn.Module,
    optimizer: optim.Optimizer,
    train_dataloader: DataLoader[Any],
    metrics: MetricCollection,
    lr_scheduler: Optional[torch.optim.lr_scheduler.LRScheduler] = None,
    binary_decision_threshold: float = 0.5,
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
) -> tuple[float, dict[str, float]]:
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_dataloader:
        X_batch, y_batch = (
            X_batch.to(device, non_blocking=True),
            y_batch.to(device, non_blocking=True).view(-1, 1).float(),
        )

        batch_loss, batch_preds = step(
            X=X_batch,
            y=y_batch,
            train=True,
            loss_fn=loss_fn,
            optimizer=optimizer,
            model=model,
            binary_decision_threshold=binary_decision_threshold,
        )

        metrics.update(batch_preds, y_batch)
        train_loss += batch_loss

    if lr_scheduler:
        lr_scheduler.step()

    train_metrics = metrics.compute()
    train_loss /= len(train_dataloader)

    metrics.reset()

    return train_loss, train_metrics


def test_loop(
    model: nn.Module,
    loss_fn: nn.Module,
    test_dataloader: DataLoader[Any],
    metrics: MetricCollection,
    binary_decision_threshold: float = 0.5,
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
) -> tuple[float, dict[str, float]]:
    test_loss = 0.0
    model.eval()
    for X_batch, y_batch in test_dataloader:
        X_batch, y_batch = (
            X_batch.to(device, non_blocking=True),
            y_batch.to(device, non_blocking=True).view(-1, 1).float(),
        )

        batch_loss, batch_preds = step(
            X=X_batch,
            y=y_batch,
            train=False,
            loss_fn=loss_fn,
            model=model,
            binary_decision_threshold=binary_decision_threshold,
        )

        metrics.update(batch_preds, y_batch)
        test_loss += batch_loss

    test_metrics = metrics.compute()
    test_loss /= len(test_dataloader)

    metrics.reset()

    return test_loss, test_metrics


def train(
    model: nn.Module,
    loss_fn: nn.Module,
    optimizer: optim.Optimizer,
    train_dataloader: DataLoader[Any],
    test_dataloader: DataLoader[Any],
    metrics: MetricCollection,
    early_stopper: Optional[EarlyStopping] = None,
    lr_scheduler: Optional[torch.optim.lr_scheduler.LRScheduler] = None,
    binary_decision_threshold: float = 0.5,
    num_epochs: int = 100,
    log_every: int = 10,
    verbose: bool = True,
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
) -> tuple[float, float]:
    """
    Trains the provided model

    ---
    Returns:
        train_loss: float:
            Training Loss
        test_loss: float:
            Testing Loss
    """
    model.to(device, non_blocking=True)

    train_loss: float
    test_loss: float
    train_metrics: dict[str, float]
    test_metrics: dict[str, float]

    train_loss, test_loss = 0.0, 0.0
    train_metrics, test_metrics = dict(), dict()

    for epoch in tqdm(range(num_epochs)):
        train_loss, train_metrics = train_loop(
            model=model,
            loss_fn=loss_fn,
            optimizer=optimizer,
            train_dataloader=train_dataloader,
            metrics=metrics,
            lr_scheduler=lr_scheduler,
            device=device,
            binary_decision_threshold=binary_decision_threshold,
        )

        test_loss, test_metrics = test_loop(
            model=model,
            loss_fn=loss_fn,
            test_dataloader=test_dataloader,
            metrics=metrics,
            device=device,
            binary_decision_threshold=binary_decision_threshold,
        )

        if verbose:
            if epoch % log_every == 0:
                train_metrics_str = " | ".join(
                    f"{' '.join(k.title().split('_'))}: {v:.2%}"
                    for k, v in train_metrics.items()
                )
                test_metrics_str = " | ".join(
                    f"{' '.join(k.title().split('_'))}: {v:.2%}"
                    for k, v in test_metrics.items()
                )
                print(
                    f"""Epoch {epoch}
Training Loss = {train_loss:.2f}\t| Testing Loss = {test_loss:.2f}
Training Metrics:
    {train_metrics_str}
Testing Metrics:
    {test_metrics_str}
__________________________________________________________________________________
"""
                )

        if early_stopper:
            early_stopper(test_loss, model)
            if early_stopper.early_stop:
                print(
                    "================================ Early Stopping ================================"
                )
                early_stopper.load_best_model(model)
                break

    train_metrics_str = " | ".join(
        f"{' '.join(k.title().split('_'))}: {v:.2%}" for k, v in train_metrics.items()
    )
    test_metrics_str = " | ".join(
        f"{' '.join(k.title().split('_'))}: {v:.2%}" for k, v in test_metrics.items()
    )
    print(
        f""" Final Results:
Training Loss = {train_loss:.2f}\t| Testing Loss = {test_loss:.2f}
Training Metrics:
    {train_metrics_str}
Testing Metrics:
    {test_metrics_str}"""
    )
    return train_loss, test_loss

# Train and test the model

## Model Initialization

In [ ]:
num_epochs: int = 5
early_stopping_patience: int = 5
early_stopping_delta: float = 0.01
num_classes: int = 3
lr: float = 1e-3
lr_scheduler_step_size: int = 10
lr_scheduler_gamma: float = 0.1
task: str = "multiclass"
binary_decision_threshold: float = 0.5

###### Set all of those to whatever suits your project, these are just examples to show how to use the train function and its components ######
loss_fn = nn.CrossEntropyLoss()
model = SimpleModel()
optimizer = optim.Adam(model.parameters(), lr=lr)

metrics = MetricCollection(
    {
        "accuracy": Accuracy(num_classes=num_classes, average="macro", task=task),
        "precision": Precision(num_classes=num_classes, average="macro", task=task),
        "recall": Recall(num_classes=num_classes, average="macro", task=task),
        "f1_score": F1Score(num_classes=num_classes, average="macro", task=task),
    }
)

early_stopper = EarlyStopping(
    patience=early_stopping_patience, delta=early_stopping_delta
)
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer=optimizer, step_size=lr_scheduler_step_size, gamma=lr_scheduler_gamma
)

h_params = {
    "learning_rate": lr,
    "optimizer": optimizer.__class__.__name__,
    "loss_function": loss_fn.__class__.__name__,
    "num_epochs": num_epochs,
    "batch_size": batch_size,
    "early_stopping_patience": early_stopping_patience,
    "early_stopping_delta": early_stopping_delta,
    "lr_scheduler_step_size": lr_scheduler_step_size,
    "lr_scheduler_gamma": lr_scheduler_gamma,
    "binary_decision_threshold": binary_decision_threshold,
}

## Train and test the model

In [ ]:
train(
    model=model,
    loss_fn=loss_fn,
    optimizer=optimizer,
    train_dataloader=train_dl,
    test_dataloader=test_dl,
    metrics=metrics,
    early_stopper=early_stopper,
    lr_scheduler=lr_scheduler,
    num_epochs=num_epochs,
    binary_decision_threshold=binary_decision_threshold,
)

## Test and log the model, data, and hyper parameters

In [ ]:
with mlflow.start_run(run_name="model_training_run"):
    mlflow.log_table(model_description_df, artifact_file="model_summary.json")
    mlflow.log_params(h_params)

    train_data = pd.concat([X_train, y_train], axis=1).reset_index(drop=True)

    dataset = mlflow.data.from_pandas(
        dataset_df, name="train-mock-dataset", targets="label"
    )
    mlflow.log_input(dataset, context="training")
    model.eval()
    signature = infer_signature(X_train, model(X_train_example).detach().cpu().numpy())
    mlflow.pytorch.log_model(
        model,
        name="torch_model",
        registered_model_name=model.__class__.__name__,
        signature=signature,
        input_example=X_train.iloc[[0]],
    )

    train_loss, train_metrics = test_loop(
        model=model, loss_fn=loss_fn, test_dataloader=train_dl, metrics=metrics
    )
    test_loss, test_metrics = test_loop(
        model=model, loss_fn=loss_fn, test_dataloader=test_dl, metrics=metrics
    )
    val_loss, val_metrics = test_loop(
        model=model, loss_fn=loss_fn, test_dataloader=val_dl, metrics=metrics
    )

    all_metrics = {
        "train_loss": train_loss,
        "test_loss": test_loss,
        "val_loss": val_loss,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"test_{k}": v for k, v in test_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    mlflow.log_metrics(all_metrics)